# ZapalloAI — Notebook 01: Exploración de Datos (EDA)

**Universidad de las Fuerzas Armadas ESPE**

Este notebook analiza el dataset ya procesado en `model/data/processed/`.

## Clases del modelo
| ID | Clase | Descripción |
|---|---|---|
| 0 | `healthy` | Hoja sana |
| 1 | `downy_mildew` | Mildiu velloso (Pseudoperonospora cubensis) |
| 2 | `leaf_curl` | Enrollamiento foliar (virus) |
| 3 | `mosaic_virus` | Virus del mosaico |
| 4 | `red_beetle` | Daño por escarabajo rojo |

In [ ]:
# ── 0. Dependencias ───────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'matplotlib', 'Pillow', 'pandas', 'seaborn'], check=False)
print('OK')

In [ ]:
# ── 1. Detectar rutas automáticamente ────────────────────────────
import os
from pathlib import Path

# Detectar la raíz del repositorio
ROOT = Path(os.path.abspath('')).resolve()
for _ in range(6):
    if (ROOT / 'model').exists() and (ROOT / 'zapallo_app').exists():
        break
    ROOT = ROOT.parent

RAW_DIR  = ROOT / 'model' / 'data' / 'raw'
PROC_DIR = ROOT / 'model' / 'data' / 'processed'
CLASSES  = ['healthy', 'downy_mildew', 'leaf_curl', 'mosaic_virus', 'red_beetle']

print(f'Raiz del repo : {ROOT}')
print(f'Dataset raw   : {RAW_DIR}')
print(f'Dataset proc  : {PROC_DIR}')

proc_ok = PROC_DIR.exists() and any(PROC_DIR.iterdir())
raw_ok  = RAW_DIR.exists()
print(f'\nDataset raw    : {"OK" if raw_ok else "NO ENCONTRADO"}')
print(f'Dataset proc   : {"OK" if proc_ok else "NO ENCONTRADO — ejecuta primero el Notebook 02"}')

In [ ]:
# ── 2. Conteo de imágenes por split y clase ───────────────────────
import pandas as pd

def count_images(base: Path):
    rows = []
    for split in ['train', 'val', 'test']:
        for cls in CLASSES:
            d = base / split / cls
            n = sum(1 for f in d.glob('*.*') if f.suffix.lower() in ['.jpg','.jpeg','.png']) if d.exists() else 0
            rows.append({'split': split, 'clase': cls, 'count': n})
    return pd.DataFrame(rows)

df = count_images(PROC_DIR)
pivot = df.pivot(index='clase', columns='split', values='count')[['train','val','test']]
pivot['TOTAL'] = pivot.sum(axis=1)
pivot.loc['TOTAL'] = pivot.sum()
print('\n=== Distribución del Dataset Procesado ===')
print(pivot.to_string())

In [ ]:
# ── 3. Gráfico de barras apiladas ─────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11})

df_plot = pivot.drop('TOTAL').drop(columns='TOTAL').reset_index()

COLORS = {'train': '#2D6A4F', 'val': '#52B788', 'test': '#95D5B2'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras apiladas
ax = axes[0]
bottom = [0] * len(CLASSES)
for split in ['train', 'val', 'test']:
    vals = [df_plot.loc[df_plot.clase==c, split].values[0] for c in CLASSES]
    ax.bar(CLASSES, vals, bottom=bottom, label=split, color=COLORS[split])
    bottom = [b + v for b, v in zip(bottom, vals)]
ax.set_title('Distribución por clase y split')
ax.set_ylabel('Imágenes')
ax.set_xticklabels(CLASSES, rotation=20, ha='right')
ax.legend()

# Solo train — balance de clases
ax2 = axes[1]
train_counts = [df_plot.loc[df_plot.clase==c, 'train'].values[0] for c in CLASSES]
bars = ax2.barh(CLASSES, train_counts, color='#2D6A4F')
for bar, val in zip(bars, train_counts):
    ax2.text(val + 30, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center')
ax2.set_title('Imágenes en TRAIN por clase')
ax2.set_xlabel('Imágenes')
ax2.invert_yaxis()

plt.suptitle('ZapalloAI — EDA del Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROC_DIR.parent / 'eda_distribucion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico guardado')

In [ ]:
# ── 4. Muestras visuales de cada clase ───────────────────────────
from PIL import Image
import numpy as np
import random

random.seed(42)
N_SAMPLES = 4  # muestras por clase

fig, axes = plt.subplots(len(CLASSES), N_SAMPLES,
                          figsize=(N_SAMPLES * 3, len(CLASSES) * 3))

for row, cls in enumerate(CLASSES):
    cls_dir = PROC_DIR / 'train' / cls
    imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
    samples = random.sample(imgs, min(N_SAMPLES, len(imgs)))

    for col in range(N_SAMPLES):
        ax = axes[row][col]
        if col < len(samples):
            img = Image.open(samples[col]).convert('RGB').resize((224, 224))
            ax.imshow(img)
            ax.set_title(cls if col == 0 else '', fontsize=9, fontweight='bold')
        ax.axis('off')

plt.suptitle('Muestras por clase (split train)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROC_DIR.parent / 'eda_muestras.png', dpi=120, bbox_inches='tight')
plt.show()
print('Muestras guardadas')

In [ ]:
# ── 5. Análisis de resoluciones ───────────────────────────────────
from PIL import Image
from collections import Counter

print('Analizando resoluciones (muestra de 100 imgs por clase)...')
res_data = []

for cls in CLASSES:
    cls_dir = PROC_DIR / 'train' / cls
    imgs = list(cls_dir.glob('*.jpg'))[:100]
    for p in imgs:
        try:
            with Image.open(p) as img:
                w, h = img.size
                res_data.append({'clase': cls, 'w': w, 'h': h, 'ratio': round(w/h, 2)})
        except Exception:
            pass

df_res = pd.DataFrame(res_data)

print(f'\nResolucion mas comun (WxH):')
res_str = df_res.apply(lambda r: f"{int(r.w)}x{int(r.h)}", axis=1)
print(Counter(res_str).most_common(5))

print(f'\nEstadisticas por clase:')
print(df_res.groupby('clase')[['w','h']].agg(['mean','min','max']).round(0).to_string())

# Scatter de resoluciones
fig, ax = plt.subplots(figsize=(8, 5))
palette = {'healthy':'#2D6A4F','downy_mildew':'#52B788','leaf_curl':'#B7E4C7',
           'mosaic_virus':'#F4A261','red_beetle':'#E76F51'}
for cls in CLASSES:
    sub = df_res[df_res.clase == cls]
    ax.scatter(sub.w, sub.h, label=cls, alpha=0.5, s=20, color=palette[cls])
ax.set_xlabel('Ancho (px)')
ax.set_ylabel('Alto (px)')
ax.set_title('Resoluciones del dataset')
ax.legend(markerscale=2)
plt.tight_layout()
plt.savefig(PROC_DIR.parent / 'eda_resoluciones.png', dpi=120)
plt.show()

In [ ]:
# ── 6. Resumen estadístico final ──────────────────────────────────
total = pivot.loc['TOTAL', 'TOTAL']
total_train = pivot.loc['TOTAL', 'train']
total_val   = pivot.loc['TOTAL', 'val']
total_test  = pivot.loc['TOTAL', 'test']

print('=== RESUMEN DATASET ZapalloAI ===')
print(f'Total imagenes  : {int(total):,}')
print(f'  Train (70%)   : {int(total_train):,}')
print(f'  Val   (15%)   : {int(total_val):,}')
print(f'  Test  (15%)   : {int(total_test):,}')
print(f'Num clases      : 5')
print()

# Balance
train_per_cls = [pivot.loc[c, 'train'] for c in CLASSES]
ratio = max(train_per_cls) / min(train_per_cls)
print(f'Clase mayor     : {CLASSES[train_per_cls.index(max(train_per_cls))]} ({int(max(train_per_cls)):,} imgs)')
print(f'Clase menor     : {CLASSES[train_per_cls.index(min(train_per_cls))]} ({int(min(train_per_cls)):,} imgs)')
print(f'Ratio max/min   : {ratio:.2f}x', '(OK)' if ratio <= 3 else '(necesita augmentation)')
print()
print('Siguiente: Notebook 02 para procesar, o Notebook 03 para entrenar')